# **Bronze Layer**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable as T

In [0]:
%run /Workspace/fmcg_pipeline/setup_fmcg_data/utilities

In [0]:
dbutils.widgets.text("catalog", "lakehouse", "Catalog")
dbutils.widgets.text("datasource", "products", "Datasource")
catlog =dbutils.widgets.get("catalog")
dataSource = dbutils.widgets.get("datasource")

## data source is a folder name ofs3 customer data , if folder name change we just change it in a widget


In [0]:
path = f's3://fmcg-child-data/{dataSource}/*.csv'
df_bronze = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path).select("*","_metadata.file_name","_metadata.file_size")
df_bronze = df_bronze.withColumn("timestamp", F.current_timestamp())

#df_bronze.printSchema()
#df_bronze.show()


In [0]:
df_bronze.write.format("delta")\
.mode("append")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{bronze_schema}.{dataSource}")

In [0]:
%sql
select * from lakehouse.bronze.products

product_name,product_id,category,file_name,file_size,timestamp
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,products.csv,1388,2026-03-01T18:54:11.764Z
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,products.csv,1388,2026-03-01T18:54:11.764Z


# **Silver Transformation Layer**

In [0]:
df_silver = df_bronze.withColumn(
    "product_id",
    F.when(df_bronze["product_id"] == "XYZ123", "25891502").otherwise(df_bronze["product_id"])
)
display(df_silver)

product_name,product_id,category,file_name,file_size,timestamp
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,products.csv,1388,2026-03-01T18:54:15.435Z
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,products.csv,1388,2026-03-01T18:54:15.435Z


In [0]:
#extract column variant 
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract("product_name", r"\((.*?)\)", 1)
)

In [0]:
# clean product name
df_silver = df_silver.withColumn(
    "product_name",
    F.regexp_replace("product_name", r"\(.*?\)", "")
)
display(df_silver)

product_name,product_id,category,file_name,file_size,timestamp,variant
SportsBar Energy Bar Choco Fudge,25891101,energy bars,products.csv,1388,2026-03-01T18:54:16.655Z,60g
SportsBar Energy Bar Choco Fudge,25891102,energy bars,products.csv,1388,2026-03-01T18:54:16.655Z,40g
SportsBar Energy Bar Choco Fudge,25891103,energy bars,products.csv,1388,2026-03-01T18:54:16.655Z,25g
SportsBar Protien Bar Peanut Crunch,25891201,protien bars,products.csv,1388,2026-03-01T18:54:16.655Z,45g
SportsBar Protien Bar Peanut Crunch,25891202,protien bars,products.csv,1388,2026-03-01T18:54:16.655Z,55g
SportsBar Protien Bar Peanut Crunch,25891203,protien bars,products.csv,1388,2026-03-01T18:54:16.655Z,65g
SportsBar Granola Crunch Honey Almond,25891301,granola & cereals,products.csv,1388,2026-03-01T18:54:16.655Z,400g
SportsBar Granola Crunch Honey Almond,25891302,granola & cereals,products.csv,1388,2026-03-01T18:54:16.655Z,300g
SportsBar Granola Crunch Honey Almond,25891303,granola & cereals,products.csv,1388,2026-03-01T18:54:16.655Z,200g
SportsBar Greek Yogurt Pro Vanilla,25891401,recovery dairy,products.csv,1388,2026-03-01T18:54:16.655Z,200g


In [0]:
df_silver = (
    df_silver
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    .withColumnRenamed("product_name", "product")
)
display(df_silver)

product,product_id,category,file_name,file_size,timestamp,variant,product_code
SportsBar Energy Bar Choco Fudge,25891101,energy bars,products.csv,1388,2026-03-01T18:54:17.684Z,60g,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
SportsBar Energy Bar Choco Fudge,25891102,energy bars,products.csv,1388,2026-03-01T18:54:17.684Z,40g,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
SportsBar Energy Bar Choco Fudge,25891103,energy bars,products.csv,1388,2026-03-01T18:54:17.684Z,25g,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
SportsBar Protien Bar Peanut Crunch,25891201,protien bars,products.csv,1388,2026-03-01T18:54:17.684Z,45g,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef
SportsBar Protien Bar Peanut Crunch,25891202,protien bars,products.csv,1388,2026-03-01T18:54:17.684Z,55g,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef
SportsBar Protien Bar Peanut Crunch,25891203,protien bars,products.csv,1388,2026-03-01T18:54:17.684Z,65g,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef
SportsBar Granola Crunch Honey Almond,25891301,granola & cereals,products.csv,1388,2026-03-01T18:54:17.684Z,400g,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217
SportsBar Granola Crunch Honey Almond,25891302,granola & cereals,products.csv,1388,2026-03-01T18:54:17.684Z,300g,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217
SportsBar Granola Crunch Honey Almond,25891303,granola & cereals,products.csv,1388,2026-03-01T18:54:17.684Z,200g,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217
SportsBar Greek Yogurt Pro Vanilla,25891401,recovery dairy,products.csv,1388,2026-03-01T18:54:17.684Z,200g,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7


In [0]:
df_silver = df_silver.withColumn("division", F.lit("Nutrition"))

In [0]:
df_silver.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{silver_schema}.{f'{dataSource}'}")

In [0]:
display(df_silver)

product,product_id,category,file_name,file_size,timestamp,variant,product_code,division
SportsBar Energy Bar Choco Fudge,25891101,energy bars,products.csv,1388,2026-03-01T18:54:47.433Z,60g,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,Nutrition
SportsBar Energy Bar Choco Fudge,25891102,energy bars,products.csv,1388,2026-03-01T18:54:47.433Z,40g,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,Nutrition
SportsBar Energy Bar Choco Fudge,25891103,energy bars,products.csv,1388,2026-03-01T18:54:47.433Z,25g,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,Nutrition
SportsBar Protien Bar Peanut Crunch,25891201,protien bars,products.csv,1388,2026-03-01T18:54:47.433Z,45g,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,Nutrition
SportsBar Protien Bar Peanut Crunch,25891202,protien bars,products.csv,1388,2026-03-01T18:54:47.433Z,55g,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,Nutrition
SportsBar Protien Bar Peanut Crunch,25891203,protien bars,products.csv,1388,2026-03-01T18:54:47.433Z,65g,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,Nutrition
SportsBar Granola Crunch Honey Almond,25891301,granola & cereals,products.csv,1388,2026-03-01T18:54:47.433Z,400g,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,Nutrition
SportsBar Granola Crunch Honey Almond,25891302,granola & cereals,products.csv,1388,2026-03-01T18:54:47.433Z,300g,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,Nutrition
SportsBar Granola Crunch Honey Almond,25891303,granola & cereals,products.csv,1388,2026-03-01T18:54:47.433Z,200g,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,Nutrition
SportsBar Greek Yogurt Pro Vanilla,25891401,recovery dairy,products.csv,1388,2026-03-01T18:54:47.433Z,200g,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,Nutrition


# **Gold Layer**

In [0]:
df_gold = df_silver.select("product_code", "division", "category", "product", "variant")
display(df_gold)

In [0]:
df_gold.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{gold_schema}.{f'sp_{dataSource}'}")


In [0]:
%sql
select * from lakehouse.gold.sp_products

# **Merge Tables **

In [0]:
delta_table = T.forName(spark, "lakehouse.gold.dim_products")
display(delta_table) 

In [0]:
df_child = spark.table(f"{catlog}.{gold_schema}.{f'sp_{dataSource}'}")
display(df_child)


In [0]:
df_child=df_child.dropDuplicates(["product_code"])


In [0]:
delta_table.alias("target").merge(
    source=df_child.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
%sql
select * from lakehouse.gold.dim_products